분석 들어갑니다.

## 1. 한 줄 진단
접근 방향(정렬 + 고정 + 투포인터)은 정확히 맞췄지만, **중복 제거 처리**라는 핵심 디테일을 구체화하지 못한 채 코드화 단계에서 멈췄습니다.

## 2. 내 사고 흐름 요약
- 문제 조건(중복 조합 제거, 같은 인덱스 중복 사용 금지, 순서 무관)을 정확히 정리했습니다.
- 브루트포스 O(n³) 대신 "한 수 고정 + 나머지 두 수 탐색"이라는 차원 축소 전략을 떠올렸습니다.
- 탐색 편의를 위해 정렬을 선행하고, 정렬된 배열의 단조성을 활용해 투포인터(left, right)로 합을 0에 맞추는 방향을 잡았습니다.
- 합이 0보다 크면 right--, 작으면 left++ 라는 이동 규칙까지 도출했습니다.
- 그러나 실제 코드는 `nums[i]` 고정 부분에서 멈췄습니다.

## 3. 막힌 이유 분석
- **첫 오판**: 없습니다. 알고리즘 골격은 정답과 동일합니다.
- **결정적으로 부족했던 점**: "중복 조합 하나만"이라는 조건을 알고리즘 레벨에서 어떻게 구현할지에 대한 **구체적 규칙**이 메모에 없었습니다. 정렬을 했다는 사실이 중복 제거에 어떻게 연결되는지(같은 값이 연속해서 나오므로 인접 비교로 스킵 가능) 까지 사고가 내려가지 않았습니다.
- **왜 여기서 막혔는지**: 큰 그림(정렬+투포인터)에서 작은 그림(i, left, right 각각에서 어떻게 중복을 건너뛸지)으로 내려가는 단계에서 추상도가 너무 높은 채로 코딩을 시작했습니다. "left, right 이동" 규칙은 정했지만, **합이 정확히 0일 때 어떻게 다음 후보로 넘어갈지**, **i를 1씩 늘릴 때 직전 값과 같으면 어떻게 할지**가 비어 있었습니다.

## 4. 실패 유형 분류
- **주 실패 유형**: 5. 구현 실패
- **부 실패 유형**: 4. 상태/불변식 설계 실패 (중복 스킵 불변식 미설계)
- **근거**: 알고리즘 선택과 복잡도 감각(O(n²))은 정확합니다. 막힌 지점은 "정렬된 배열에서 같은 값이 연속한다"는 정렬의 부수 효과를 중복 제거 메커니즘으로 연결하지 못한 점, 그리고 left/right가 정답을 찾은 직후 어디로 이동해야 하는지 정하지 못한 점입니다.

## 5. 등급 판정
- **판정: B**
- **이유**: 자력으로 큰 골격을 짰고, 중복 처리 규칙 하나(예: "i, left, right 각각에서 직전 값과 같으면 스킵")만 힌트로 받으면 충분히 완성할 수 있는 상태입니다.

=> 내가 판단했을 땐 D

## 6. 정답 풀이에서 뽑아낼 일반화 포인트

**자료구조/알고리즘 포인트**
정렬 + 투포인터는 "정렬된 배열에서 두 수의 합이 특정 값이 되는 쌍을 O(n)에 찾는" 표준 도구입니다. k-Sum 계열은 보통 "k-2개를 고정하고 나머지 2개에 대해 투포인터"로 차원을 줄입니다. 3Sum은 그 가장 단순한 사례입니다. 일반적으로 외부 루프 1개 + 내부 투포인터로 O(n²)이 됩니다.

**상태/불변식 포인트**
정렬된 배열에서 투포인터의 핵심 불변식은 두 가지입니다.
첫째, `nums[left] + nums[right]`는 left를 늘리면 단조 증가, right를 줄이면 단조 감소합니다. 이 단조성 덕분에 한쪽 방향으로만 움직여도 모든 의미 있는 후보를 빠짐없이 탐색합니다.
둘째, 중복 제거 불변식: 정렬된 배열에서 **같은 값은 반드시 연속**하므로, "이전 인덱스 값과 같으면 스킵"이라는 단 한 줄로 중복 조합을 막을 수 있습니다. 이 규칙은 i, left, right **세 군데 모두**에 적용해야 합니다 (i는 외부 루프에서, left/right는 합이 0인 답을 찾은 직후 다음 위치로 이동할 때).

**복잡도/경계조건 포인트**
- 정렬 O(n log n) + 외부 루프 O(n) × 내부 투포인터 O(n) = **O(n²)**. n ≤ 3000 정도에서 충분히 통과합니다.
- 조기 종료: `nums[i] > 0`이면 정렬된 배열에서 뒤따라오는 값들도 모두 양수라 합이 0이 될 수 없으므로 `break`. 작은 최적화지만 사고 훈련에 좋습니다.
- 경계: `i`의 범위는 `0 ≤ i ≤ n-3`, `left = i+1`, `right = n-1`로 시작. left < right 동안만 반복.
- 답을 찾은 직후 left++와 right-- **둘 다** 해야 합니다 (한쪽만 움직이면 같은 합이 또 나옴).

**다른 문제에 적용할 수 있는 일반화 문장**
"정렬된 배열에서 합/차이/조건을 만족하는 쌍이나 조합을 찾을 때, 단조성을 이용한 투포인터를 우선 검토하고, 중복 조합 제거가 필요하면 '직전 값과 같으면 스킵'을 i/left/right 각 지점마다 따로 적용한다."

## 7. 다음에 써먹을 트리거 문장
- "정렬된 배열에서 두 수/세 수의 합이 X" → 정렬 + 투포인터, k-Sum은 (k-2)중 루프 + 투포인터
- "중복 조합 제거 + 정렬 사용 가능" → "직전 인덱스와 값이 같으면 continue" 3종 세트 (외부 i, 내부 left, 내부 right)
- "합이 너무 크면/작으면 한쪽 줄이고 한쪽 늘린다" → 정렬된 배열의 단조성 활용 신호
- "정답 한 쌍을 찾은 직후 다음 후보로 이동" → left++와 right-- 동시 적용 + 중복 스킵

## 8. 개선 액션
- **오늘 바로 할 것**: 3Sum을 정렬 + 투포인터 + 3중 중복 스킵으로 백지 구현하고, "왜 left/right 둘 다 움직여야 하는지", "왜 `i > 0 and nums[i] == nums[i-1]` 조건에 `i > 0`이 필요한지"를 주석으로 직접 적어보기.
- **내일 복습할 것**: 같은 패턴의 LeetCode 16. 3Sum Closest, 18. 4Sum 풀기. 4Sum에서 외부 이중 루프 + 투포인터로 자연스럽게 확장되는지 확인.
- **비슷한 문제에서 확인할 포인트**: "중복 입력이 허용되는데 결과에서는 중복 제거가 요구되는가?" 이 질문을 문제 해석 단계에서 먼저 던지고, 답이 Yes면 정렬 + 인접 비교 스킵을 처음부터 설계에 포함시키기.

## 9. 오답노트용 요약
- **등급**: B
- **유형**: 구현 실패 (+ 부분적 상태 설계 실패)
- **막힌 이유**: 정렬+투포인터 골격은 잡았으나 중복 제거 메커니즘과 정답 발견 후 포인터 이동 규칙을 구체화하지 못함
- **일반화 포인트**: k-Sum = (k-2)중 루프 + 정렬된 배열에서의 투포인터, 중복 제거는 "직전 값과 같으면 스킵"을 모든 인덱스 변수에 적용
- **트리거**: "정렬 가능 + 합 조건 + 중복 조합 제거" → 정렬 + 투포인터 + 3종 스킵
- **다음 액션**: 3Sum 백지 구현 → 3Sum Closest, 4Sum으로 확장 훈련

주어진 숫자 배열에서 세 원소를 조합했을 때 
같은 원소를 중복 사용하지 않으며, 원소의 합이 0인 배열을 반환 

단 중복되는 조합은 하나만 선택
순서상관 x

조건에 만족하는 원소 조합이 없으면 [] 반환

숫자하나를 고정
나머지 두원소 를 조합해가며 조건을 만족하는지 탐색


[-1,0,1,2,-1,-4]

탐색 편의를 위해 정렬
[-4,-1,-1,0,1,2]

두수를 left, right 으로 설정
남은 두수와 고정된 수를 더해서 0보다 크면
right를 -=1

작으면  left += 1

class Solution:
    def threeSum(self, nums: list[int]) -> list[list[int]]:
        숫자 하나를 선택 nums[i]
